In [17]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [18]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [19]:
# SETUP & QUANTUM RANDOM NUMBER GENERATOR
num_bits = 50
simulator = BasicSimulator()

def generate_quantum_random_bits(length):
    """Generates random bits by measuring a qubit in superposition."""
    bits = []
    for _ in range(length):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        t_qc = transpile(qc, simulator)
        result = simulator.run(t_qc, shots=1).result()
        bit = int(list(result.get_counts().keys())[0])
        bits.append(bit)
    return bits

In [20]:
# ALICE'S ACTIONS
print("--- ALICE ---")
alice_bits = generate_quantum_random_bits(num_bits)
alice_bases = generate_quantum_random_bits(num_bits)

print(f"Alice's bits:  {alice_bits}")
print(f"Alice's bases: {alice_bases}")

encoded_qubits = []
for i in range(num_bits):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1:
        qc.x(0)
    if alice_bases[i] == 1:
        qc.h(0)
    encoded_qubits.append(qc)

print("Alice encoded and sent her qubits.")

--- ALICE ---
Alice's bits:  [0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1]
Alice's bases: [1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1]
Alice encoded and sent her qubits.


In [21]:
# EVE'S ACTIONS (THE ATTACKER)
print("\n--- EVE (ATTACKER) ---")
# Eve intercepts the qubits and guesses the bases
eve_bases = generate_quantum_random_bits(num_bits)
eve_bits = []
qubits_for_bob = []

for i in range(num_bits):
    qc = encoded_qubits[i]

    # Eve measures the intercepted qubit
    if eve_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)

    t_qc = transpile(qc, simulator)
    result = simulator.run(t_qc, shots=1).result()
    measured_bit = int(list(result.get_counts().keys())[0])
    eve_bits.append(measured_bit)

    # Eve prepares a NEW qubit to send to Bob based on her result
    new_qc = QuantumCircuit(1, 1)
    if measured_bit == 1:
        new_qc.x(0)
    if eve_bases[i] == 1:
        new_qc.h(0)
    qubits_for_bob.append(new_qc)

print(f"Eve's bases:   {eve_bases}")
print(f"Eve's bits:    {eve_bits}")
print("Eve intercepted the qubits, measured them, and forwarded new ones to Bob.")


--- EVE (ATTACKER) ---
Eve's bases:   [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1]
Eve's bits:    [0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1]
Eve intercepted the qubits, measured them, and forwarded new ones to Bob.


In [22]:
# BOB'S ACTIONS
print("\n--- BOB ---")
bob_bases = generate_quantum_random_bits(num_bits)
bob_bits = []

# Bob measures the qubits he received (which are actually from Eve)
for i in range(num_bits):
    qc = qubits_for_bob[i]
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)

    t_qc = transpile(qc, simulator)
    result = simulator.run(t_qc, shots=1).result()
    measured_bit = int(list(result.get_counts().keys())[0])
    bob_bits.append(measured_bit)

print(f"Bob's bases:   {bob_bases}")
print(f"Bob's bits:    {bob_bits}")
print("Bob received and measured the qubits.")


--- BOB ---
Bob's bases:   [0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0]
Bob's bits:    [0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1]
Bob received and measured the qubits.


In [23]:
# PUBLIC COMMUNICATION & SIFTING
print("\n--- PUBLIC COMMUNICATION ---")
sifted_key_alice = []
sifted_key_bob = []

# Alice and Bob sift their keys based on matching bases
for i in range(num_bits):
    if alice_bases[i] == bob_bases[i]:
        sifted_key_alice.append(alice_bits[i])
        sifted_key_bob.append(bob_bits[i])
print(f"Sifted Key Length: {len(sifted_key_alice)} bits remaining (out of {num_bits})")


--- PUBLIC COMMUNICATION ---
Sifted Key Length: 23 bits remaining (out of 50)


In [24]:
# ERROR CHECKING & THRESHOLD DETECTION
sample_size = len(sifted_key_alice) // 2
sample_alice = sifted_key_alice[:sample_size]
sample_bob = sifted_key_bob[:sample_size]

errors = sum(1 for a, b in zip(sample_alice, sample_bob) if a != b)
error_rate = errors / sample_size if sample_size > 0 else 0

# Set an acceptable error threshold (in an ideal world, 0%, but we set to > 10% to flag an attack)
ERROR_THRESHOLD = 0.10

print(f"Error check on {sample_size} bits.")
print(f"Errors found: {errors}")
print(f"Error Rate: {error_rate * 100:.2f}%")

if error_rate > ERROR_THRESHOLD:
    print("\n ATTACK DETECTED!")
    print(f"The error rate ({error_rate * 100:.2f}%) exceeds the threshold of {ERROR_THRESHOLD * 100}%.")
    print("Protocol aborted. The quantum channel is compromised.")
else:
    print("\nSuccess! No eavesdropper detected")

Error check on 11 bits.
Errors found: 4
Error Rate: 36.36%

 ATTACK DETECTED!
The error rate (36.36%) exceeds the threshold of 10.0%.
Protocol aborted. The quantum channel is compromised.
